In [1]:
from random import sample
import json
import base64
import gzip
from io import BytesIO
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from sqlalchemy import select, func
from eyened_orm import (
    Creator,
    ImageInstance,
    Modality,
    Feature,
    Annotation,
    AnnotationData,
    AnnotationType,
    Segmentation
)
from eyened_orm.db import Database
from eyened_orm.utils.attributes import df_to_attributes, print_import_summary

In [2]:
database = Database('../dev/eyened_dev.env')
session = database.create_session()

In [3]:
df = pd.read_csv('/mnt/oogergo/eyened/vasculature/releases/08_25/release_full.csv', index_col=0, nrows=100)

In [4]:
df.head()

,bif_angles_arteries_mean,bif_angles_arteries_median,bif_angles_veins_mean,bif_angles_veins_median,bif_arteries,bif_veins,coverage_vessels,cre_arteries,cre_veins,diam_arteries_median,...,tort_vessels_infl_arteries,tort_vessels_infl_veins,tort_vessels_skl_arteries,tort_vessels_skl_veins,tort_vessels_spl_arteries,tort_vessels_spl_veins,vd_arteries,vd_total_arteries,vd_total_veins,vd_veins
193024,81.556199,86.443819,84.580285,84.796144,21.0,22.0,0.038545,10.891064,15.698664,4.046173,...,2.0,2.0,1.094359,1.090123,1.032671,1.027740,0.054681,0.049093,0.048473,0.064103
193027,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193032,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
193035,NaN,NaN,83.103916,83.103916,0.0,1.0,0.378599,NaN,2.288663,5.446048,...,1.0,0.0,1.158825,1.056768,1.095385,1.011553,NaN,0.000483,0.001393,NaN
193038,80.726368,82.539068,80.016493,81.128589,23.0,27.0,0.035730,10.605799,15.280858,3.234918,...,1.5,2.0,1.088264,1.085257,1.021992,1.031694,0.056898,0.048109,0.050957,0.061723


In [5]:
# df should have image IDs in its index and attributes as columns
image_attributes = df_to_attributes(session, df, model_name="VascX", version="0.1")

# Optional: print summary of what was imported
attributes = [ia.Attribute for ia in image_attributes if ia.Attribute is not None]
print_import_summary(attributes, image_attributes)

New Attributes:
  - bif_angles_arteries_mean: 95 inserted
  - bif_angles_arteries_median: 95 inserted
  - bif_angles_veins_mean: 96 inserted
  - bif_angles_veins_median: 96 inserted
  - bif_arteries: 96 inserted
  - bif_veins: 96 inserted
  - coverage_vessels: 96 inserted
  - cre_arteries: 95 inserted
  - cre_veins: 96 inserted
  - diam_arteries_median: 96 inserted
  - diam_arteries_std: 96 inserted
  - diam_veins_median: 96 inserted
  - diam_veins_std: 96 inserted
  - lapl_image: 96 inserted
  - norm_tort_segments_curv_arteries: 96 inserted
  - norm_tort_segments_curv_veins: 96 inserted
  - norm_tort_segments_infl_arteries: 96 inserted
  - norm_tort_segments_infl_veins: 96 inserted
  - norm_tort_segments_skl_arteries: 96 inserted
  - norm_tort_segments_skl_veins: 96 inserted
  - norm_tort_segments_spl_arteries: 96 inserted
  - norm_tort_segments_spl_veins: 96 inserted
  - odfd_image: 4 inserted
  - ta_arteries: 95 inserted
  - ta_veins: 95 inserted
  - temp_angle_arteries: 92 inserted

In [6]:
for attribute in attributes:
    session.add(attribute)
# # Don't forget to commit the transaction
session.commit()